Metrics
- **CER** — Character Error Rate

    > Prompting strategy: Zero-shot only instruction provided in English and no example will be provided.

    > Model: Experiments are run via the shared UVA server. See Section 1 for setup.

## Section 1 — Setup


In [115]:
import os, time, random
import editdistance
import pandas as pd
from pathlib import Path
import openai
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from datasets import load_dataset

In [ ]:
env_path = os.path.join(os.getcwd(), ".env")
load_dotenv(dotenv_path=env_path, override=True)
load_dotenv(dotenv_path=".env", override=True)

client = openai.OpenAI(
    api_key=os.environ["SURF_API"].strip(),
    base_url="https://llmproxy.uva.nl",
    timeout=30.0
)

In [117]:
client

In [118]:
# Check available models on the survers 
models = client.models.list()
for m in models.data:
    print(m.id)
# Available models - 
# Qwen-2.5-VL
# mistral-small-3.2
# gpt-oss-120b
# default-image
# whisper

Qwen-2.5-VL
mistral-small-3.2
gpt-oss-120b
default-image
whisper


In [119]:
MODEL = "mistral-small-3.2"  # this the smallest so the fastest to run our pilot test

WORKING_PATH = 'T1_glotlid'  # directory that stores output files
os.makedirs(WORKING_PATH, exist_ok=True)

OUTPUT_FILE_NAME = 'Task_pilot_result.csv'
OUT_PATH = os.path.join(WORKING_PATH, OUTPUT_FILE_NAME)

# HuggingFace dataset config
HF_DATASET = "cis-lmu/GlotLID"
HF_SPLIT   = "train"
HF_TOKEN   = os.environ["HF_TOKEN"]   # set in .env — required for gated dataset access

# Languages to evaluate (use GlotLID config codes)
SELECTED_LANG = ['fra_Latn', 'deu_Latn', 'cmn_Hani', 'jpn_Jpan', 'bod_Tibt', 'hak_Latn', 'hak_Hani', 'gla_Latn']

# How many rows to load per language (set to None to load all)
SAMPLES_PER_LANG = 100

#  Experiment parameters
WORKERS     = 5    # parallel requests
SAVE_EVERY  = 25   # save intermediate results
MAX_RETRIES = 6
TEMPERATURE = 0.0  # deterministic output

CER_THRESHOLD = 0.00  # a row passes only if output is a perfect copy

# ── Connectivity test
try:
    test = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Reply with the single word: connected"}],
        max_tokens=5, temperature=0,
    )
    print("✓ Proxy connected:", test.choices[0].message.content.strip())
except Exception as e:
    print(f"✗ Connection failed: {e}")

✓ Proxy connected: connected


In [ ]:
from datasets import get_dataset_config_names
labels = get_dataset_config_names("cis-lmu/GlotLID-Corpus") 

## Section 2 — Dataset Loading

Data is streamed directly from HuggingFace (`cis-lmu/GlotLID`). Each language is a separate dataset config. The rows are collected into a DataFrame with the following columns:

| Column | Description |
|---|---|
| `config` | Language/script code (e.g. `eng_Latn`, `pan_Guru`) |
| `text` | The source text the LLM must copy back exactly |

In [120]:
rows = []
for lang in SELECTED_LANG:
    ds = load_dataset(
        HF_DATASET, lang,
        split=HF_SPLIT,
        streaming=True,
        trust_remote_code=True,
        token=HF_TOKEN,
    )
    sample = ds.take(SAMPLES_PER_LANG) if SAMPLES_PER_LANG else ds
    for row in sample:
        rows.append({"config": lang, "text": row["text"]})

df = pd.DataFrame(rows).reset_index(drop=True)
print(f"Dataset size      : {len(df)}  ({SAMPLES_PER_LANG or 'all'} rows × {len(SELECTED_LANG)} languages)")
print(f"Languages included: {sorted(df['config'].unique())}")
df.head()

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cis-lmu/GlotLID' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cis-lmu/GlotLID' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cis-lmu/GlotLID' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Huggi

Dataset size      : 800  (100 rows × 8 languages)
Languages included: ['bod_Tibt', 'cmn_Hani', 'deu_Latn', 'fra_Latn', 'gla_Latn', 'hak_Hani', 'hak_Latn', 'jpn_Jpan']


,config,text
0,fra_Latn,C'est ma famille. C'est mon père. C'est ma mèr...
1,fra_Latn,J'ai deux enfants. Mon fils a cinq ans. Ma fil...
2,fra_Latn,"Tout d'abord, il balaie le sol avec un balai. ..."
3,fra_Latn,Puis il prend un seau d'eau et une serpillère....
4,fra_Latn,Il vide la poubelle et y met un nouveau sac po...


## Section 3 —  Prompt Design


The copy-back prompt uses a system + user message pattern:

- **System message** — sets the model's role as a strict repeater
- **User message** — contains the instruction and the text to copy

The model output is then cleaned to strip common LLM prefixes that indicate the model did not follow instructions.

In [121]:
def build_messages(text: str) -> list:
    """
    Zero-shot copy-back prompt using system + user messages.
    The system message constrains the model's behaviour;
    the user message provides the copy instruction and the text.
    """
    system_msg = (
        "You are a text-repeating machine. "
        "Output ONLY the exact input text. "
        "Do not add intro, explanation, quotes."
    )
    user_msg = (
        f"Repeat the following text exactly, character for character, with no changes: {text}\n"
        "Do not add any additional text, explanation, or punctuation."
    )
    return [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]


# Common LLM prefixes that indicate non-compliance — strip these from output
STRIP_PREFIXES = [
    "Repeat this exactly:",
    "Repeat the following text exactly, character for character, with no changes:",
    "Sure!",
    "Here is the text:",
    "Here's the text:",
]

def clean_output(text: str) -> str:
    """Remove known LLM non-compliance prefixes from the output."""
    for prefix in STRIP_PREFIXES:
        text = text.removeprefix(prefix)
    return text.strip()


# ── Verify prompt format ───────────────────────────────────────────────────────
test_text = "aap kaise ho tussi dasso"
for m in build_messages(test_text):
    print(f"[{m['role']:7}] {m['content']}")

[system ] You are a text-repeating machine. Output ONLY the exact input text. Do not add intro, explanation, quotes.
[user   ] Repeat the following text exactly, character for character, with no changes: aap kaise ho tussi dasso
Do not add any additional text, explanation, or punctuation.


## Section 4 — Metric Computation

In [122]:
def compute_cer(reference: str, output: str) -> float:
    """
    Character Error Rate — edit distance over raw character sequences.
    Matches the reference implementation exactly.
    """
    if len(reference) == 0:
        return 0.0
    return editdistance.eval(reference, output) / len(reference)


def add_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """Recompute CER and Pass/Fail from reference 'text' and model 'output'."""
    df = df.copy()
    df["cer"] = df.apply(lambda r: round(compute_cer(r["text"], r["output"]), 4), axis=1)
    df["pf"]  = df["cer"].apply(lambda c: "F" if c > CER_THRESHOLD else "P")
    return df

## Section 5 — Running the Experiment

In [123]:
def call_llm(text: str) -> str:
    """
    Send a copy-back request to the proxy.
    Retries on RateLimitError with exponential back-off + jitter.
    Returns cleaned model output, or an ERROR string on failure.
    """
    messages = build_messages(text)
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                messages=messages,
            )
            raw    = response.choices[0].message.content
            return clean_output(raw)
        except openai.RateLimitError:
            wait = (2 ** attempt) + random.uniform(0, 2)
            time.sleep(wait)
        except Exception as e:
            return f"ERROR: {e}"
    return "ERROR: rate limit — max retries exceeded"

print("✓ call_llm() defined")

✓ call_llm() defined


In [124]:
def checkpoint_path() -> str:
    return os.path.join(WORKING_PATH, f"checkpoint_{MODEL.replace(':','-')}.csv")

def load_checkpoint() -> dict:
    """Load previously completed rows (idx → (output, cer, pf))."""
    path = checkpoint_path()
    if os.path.exists(path):
        ckpt    = pd.read_csv(path, keep_default_na=False)
        results = {int(r["idx"]): (r["output"], float(r["cer"]), r["pf"])
                   for _, r in ckpt.iterrows()}
        print(f"  ↩ Resuming from checkpoint: {len(results)} rows already done.")
        return results
    return {}

def save_checkpoint(results: dict):
    """Save current results dict to a checkpoint CSV."""
    rows = [{"idx": i, "output": o, "cer": c, "pf": p}
            for i, (o, c, p) in results.items()]
    pd.DataFrame(rows).to_csv(checkpoint_path(), index=False)

In [125]:
def process_row(idx: int, text: str) -> tuple:
    """Process a single row: call LLM, compute CER, assign Pass/Fail."""
    output = call_llm(text)
    cer    = compute_cer(text, output)
    pf     = "F" if cer > CER_THRESHOLD else "P"
    return idx, output, round(cer, 4), pf


def run_experiment(df: pd.DataFrame) -> pd.DataFrame:
    """
    Run the copy-back experiment using parallel workers.
    Saves checkpoints every SAVE_EVERY completions so progress is not lost.
    Skips rows already completed in a previous run.
    """
    results   = load_checkpoint()
    todo      = [(i, row["text"]) for i, row in df.iterrows() if i not in results]
    total     = len(df)
    completed = len(results)

    print(f" Model   : {MODEL}")
    print(f" To do   : {len(todo)} / {total} rows")

    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = {executor.submit(process_row, i, s): i for i, s in todo}
        for future in as_completed(futures):
            try:
                idx, output, cer, pf = future.result(timeout=120)
            except Exception as e:
                i = futures[future]
                results[i] = (f"ERROR: {e}", 1.0, "F")
                completed += 1
                continue

            results[idx] = (output, cer, pf)
            completed    += 1

            if completed % SAVE_EVERY == 0:
                save_checkpoint(results)

            # ASCII progress bar
            pct    = completed / total
            filled = int(30 * pct)
            bar    = "█" * filled + "░" * (30 - filled)
            passes = sum(1 for _, _, p in results.values() if p == "P")
            print(
                f"\r  [{bar}] {completed}/{total} ({pct*100:.0f}%) | "
                f"Pass: {passes/completed*100:.1f}%",
                end="", flush=True,
            )

    save_checkpoint(results)
    print()

    out_df = df.copy()
    out_df["output"] = [results[i][0] for i in df.index]
    out_df["cer"]    = [results[i][1] for i in df.index]
    out_df["pf"]     = [results[i][2] for i in df.index]
    out_df["model"]  = MODEL

    out_df.to_csv(OUT_PATH, index=False)
    avg_cer   = out_df["cer"].mean()
    pass_rate = (out_df["pf"] == "P").mean() * 100
    print(f"  Avg CER: {avg_cer:.4f} | Pass rate: {pass_rate:.1f}%")
    print(f"  Saved → {OUT_PATH}")

    # Remove checkpoint after successful run
    if os.path.exists(checkpoint_path()):
        os.remove(checkpoint_path())

    return out_df

## Section 6 — Sample Run (Dummy Output)

- The cell below shows what your `results_df` should look like after running the experiment.  
Use this to verify your pipeline before running the full the actual dataset.

In [126]:
# ── Dummy results for pipeline verification ───────────────────────────────────
dummy_data = [
    {"config": "eng_Latn", "text": "The quick brown fox jumps over the lazy dog",
     "model": "Mistral-small",
     "output": "The quick brown fox jumps over the lazy dog",
     "cer": 0.000},

    # English — silent normalization (model lowercases)
    {"config": "eng_Latn", "text": "Do NOT modify this text.",
     "model": "Mistral-small",
     "output": "Do not modify this text.",
     "cer": 0.033, "pf": "F"},

    # Punjabi Romanized — perfect copy
    {"config": "pan_Latn", "text": "aap kaise ho tussi dasso",
     "model": "Mistral-small",
     "output": "aap kaise ho tussi dasso",
     "cer": 0.000},

    # Punjabi Romanized — prompt interference (model adds translation)
    {"config": "pan_Latn", "text": "main ghar ja raha haan",
     "model": "Mistral-small",
     "output": "main ghar ja raha haan (I am going home)",
     "cer": 0.481},

    # Punjabi Gurmukhi — perfect copy
    {"config": "pan_Guru", "text": "ਤੁਸੀਂ ਕਿਵੇਂ ਹੋ",
     "model": "Mistral-small",
     "output": "ਤੁਸੀਂ ਕਿਵੇਂ ਹੋ",
     "cer": 0.000},

    # Punjabi Gurmukhi — cross-script substitution (switched to Shahmukhi)
    {"config": "pan_Guru", "text": "ਚੰਗਾ ਕੰਮ ਕਰੋ",
     "model": "Mistral-small",
     "output": "چنگا کم کرو",
     "cer": 1.000},

    # Amharic — perfect copy
    {"config": "amh_Ethi", "text": "እንዴት ናቸው",
     "model": "Mistral-small",
     "output": "እንዴት ናቸው",
     "cer": 0.000},

    # Amharic — deletion
    {"config": "amh_Ethi", "text": "ጤና ይስጥልኝ እንዴት ናቸው",
     "model": "Mistral-small",
     "output": "ጤና ይስጥልኝ",
     "cer": 0.400, "pf": "F"},
]

dummy_df = pd.DataFrame(dummy_data)
print(f"Dummy dataset: {len(dummy_df)} rows | Pass: {(dummy_df['pf']=='P').sum()} | Fail: {(dummy_df['pf']=='F').sum()}")
dummy_df[["config","text","output","cer","pf"]]

Dummy dataset: 8 rows | Pass: 0 | Fail: 2


,config,text,output,cer,pf
0,eng_Latn,The quick brown fox jumps over the lazy dog,The quick brown fox jumps over the lazy dog,0.000,NaN
1,eng_Latn,Do NOT modify this text.,Do not modify this text.,0.033,F
2,pan_Latn,aap kaise ho tussi dasso,aap kaise ho tussi dasso,0.000,NaN
3,pan_Latn,main ghar ja raha haan,main ghar ja raha haan (I am going home),0.481,NaN
4,pan_Guru,ਤੁਸੀਂ ਕਿਵੇਂ ਹੋ,ਤੁਸੀਂ ਕਿਵੇਂ ਹੋ,0.000,NaN
5,pan_Guru,ਚੰਗਾ ਕੰਮ ਕਰੋ,چنگا کم کرو,1.000,NaN
6,amh_Ethi,እንዴት ናቸው,እንዴት ናቸው,0.000,NaN
7,amh_Ethi,ጤና ይስጥልኝ እንዴት ናቸው,ጤና ይስጥልኝ,0.400,F


In [127]:
# ── Verify on dummy data ──────────────────────────────────────────────────────
dummy_df = add_metrics(dummy_df)
dummy_df[["config","text","output","cer","pf"]]

,config,text,output,cer,pf
0,eng_Latn,The quick brown fox jumps over the lazy dog,The quick brown fox jumps over the lazy dog,0.0000,P
1,eng_Latn,Do NOT modify this text.,Do not modify this text.,0.1250,F
2,pan_Latn,aap kaise ho tussi dasso,aap kaise ho tussi dasso,0.0000,P
3,pan_Latn,main ghar ja raha haan,main ghar ja raha haan (I am going home),0.8182,F
4,pan_Guru,ਤੁਸੀਂ ਕਿਵੇਂ ਹੋ,ਤੁਸੀਂ ਕਿਵੇਂ ਹੋ,0.0000,P
5,pan_Guru,ਚੰਗਾ ਕੰਮ ਕਰੋ,چنگا کم کرو,0.8333,F
6,amh_Ethi,እንዴት ናቸው,እንዴት ናቸው,0.0000,P
7,amh_Ethi,ጤና ይስጥልኝ እንዴት ናቸው,ጤና ይስጥልኝ,0.5294,F


In [128]:
def summarise_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """Avg CER and pass rate grouped by language (config)."""
    grp = df.groupby("config")
    return pd.DataFrame({
        "Avg CER"    : grp["cer"].mean().round(4),
        "Pass Rate"  : grp["pf"].apply(lambda s: f"{(s=='P').mean()*100:.1f}%"),
        "Pass Count" : grp["pf"].apply(lambda s: (s=="P").sum()),
        "Fail Count" : grp["pf"].apply(lambda s: (s=="F").sum()),
        "N"          : grp["cer"].count(),
    }).reset_index().sort_values("Avg CER")

print("── Summary by language ──")
summarise_metrics(dummy_df)

── Summary by language ──


,config,Avg CER,Pass Rate,Pass Count,Fail Count,N
1,eng_Latn,0.0625,50.0%,1,1,2
0,amh_Ethi,0.2647,50.0%,1,1,2
3,pan_Latn,0.4091,50.0%,1,1,2
2,pan_Guru,0.4166,50.0%,1,1,2


In [129]:
results_df = run_experiment(df)
results_df.head()

 Model   : mistral-small-3.2
 To do   : 800 / 800 rows
  [██████████████████████████████] 800/800 (100%) | Pass: 92.2%
  Avg CER: 0.0075 | Pass rate: 92.2%
  Saved → T1_glotlid/Task_pilot_result.csv


,config,text,output,cer,pf,model
0,fra_Latn,C'est ma famille. C'est mon père. C'est ma mèr...,C'est ma famille. C'est mon père. C'est ma mèr...,0.0,P,mistral-small-3.2
1,fra_Latn,J'ai deux enfants. Mon fils a cinq ans. Ma fil...,J'ai deux enfants. Mon fils a cinq ans. Ma fil...,0.0,P,mistral-small-3.2
2,fra_Latn,"Tout d'abord, il balaie le sol avec un balai. ...","Tout d'abord, il balaie le sol avec un balai. ...",0.0,P,mistral-small-3.2
3,fra_Latn,Puis il prend un seau d'eau et une serpillère....,Puis il prend un seau d'eau et une serpillère....,0.0,P,mistral-small-3.2
4,fra_Latn,Il vide la poubelle et y met un nouveau sac po...,Il vide la poubelle et y met un nouveau sac po...,0.0,P,mistral-small-3.2


In [130]:
pass_rates = (results_df.groupby("config")["pf"].apply(lambda x: (x == "P").mean()) * 100)
print("── Pass rates by language ──")
print(pass_rates.sort_values(ascending=False).to_string())

── Pass rates by language ──
config
deu_Latn    99.0
jpn_Jpan    99.0
cmn_Hani    95.0
hak_Hani    92.0
gla_Latn    91.0
bod_Tibt    90.0
hak_Latn    88.0
fra_Latn    84.0


## Section 7 — Error Category Analysis

For each model output that has CER > 0.0, classify the error(s) into one or more of the T1 error categories. You can also create a new error catagories if you observe errors not included in the existing catagories.

| Category | Level | Detection Strategy |
|---|---|---|
| Deletion | Character | Reference Character absent in output |
| Insertion | Character | Extra Character in output not in reference |
| substitution | Character | Character replaced other characters |